Conducted by: Maddie Bomsta

#Problem and Discovery

Business Problem:

A hospital network of 130 locations wants to identify diabetic patients who are at elevated risk of readmission within 30 days so that care teams can review potential risk factors before discharge.

Current Data Enviroment:

* Historical clinical data originates from hospital EHR systems.

* The dataset represents 130 U.S. hospitals/integrated delivery networks.

* The available dataset contains approximately 101,000 encounters and structured demographic, utilization, diagnosis, medication, and hospital-stay information.

* Several fields contain substantial missing values and require explicit treatment.

Clinical Workflow Gap:

A prediction alone does not provide sufficient context for a clinician.

Client Concerns:

Patients are not receiving preventive and theraputic interventions because of arbitruary diagnosis management in the hospitals.

Opportunity from FDE:

The opportunity is to build a grounded clinical decision-support workflow that connects the predictive model, patient context, current care evidence, care gaps, clinician explanationm and safety review.

Success Criteria:

The prototype demonstrates that the system can:

1. Produce a reproducible readmission-risk score.

2. Explain the major model-associated factors.

3. Retrieve relevant information from current diabetes-care guidance.

4. Identify potential areas for clinician review.

5. Ground generated recommendations in retrieved evidence.

6. Provide an auditable output that a clinician can review rather than an autonomous medical decision.

#Model Creation and Evaluation

In [301]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, cohen_kappa_score
import joblib

raw_ehr = pd.read_csv("/content/diabetic_data.csv")
raw_ehr.head(5)
# raw_ehr.shape # 38,652 rows and 50 columns (features)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [302]:
# Saving raw data for agent needs
f = (
    raw_ehr
    .sort_values("encounter_id")
    .drop_duplicates(
        subset="patient_nbr",
        keep="first"
    )
    .copy()
)

print("Patient-level modeling data:", f.shape)

# Saving identifiers
patient_ids = f[["patient_nbr", "encounter_id"]].copy()

Patient-level modeling data: (71518, 50)


In [303]:
# 1. Cleaning Data

def clean_diabetes_data(df):

  """ Cleans and prepares the diabetes dataset for modeling.
  Applies the same cleaning steps to new EHR data uploaded. """

  df = df.copy()

# Drop hospice cases
  expired_hospice = [11, 13, 14, 19, 20, 21]
  df = df[~df['discharge_disposition_id'].isin(expired_hospice)].copy()

# Drop duplicate encounters identifiers
  df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first').copy()

# Drop near-constant medication - noise reduction
  drop_cols = [
      'encounter_id','patient_nbr','weight','payer_code','medical_specialty',
      'diag_2', 'diag_3',
      'chlorpropamide','acetohexamide','tolbutamide','acarbose','miglitol','troglitazone',
      'tolazamide','examide','citoglipton','glipizide-metformin','glimepiride-pioglitazone',
      'metformin-rosiglitazone','metformin-pioglitazone'
  ]
  df = df.drop(columns=drop_cols)

# Recode age to midpoint - can treat as continuous now
  age_map = {f"[{i}-{i+10})": i + 5 for i in range(0, 100, 10)}
  df['age'] = df['age'].map(age_map)

# Recode diagnosis 1 code into the 9 clinical groups - saves from one-hot encoding 700 raw codes
  def icd9_group(code):
      if pd.isna(code) or code == '?':
          return 'Missing'
      try:
          if str(code).startswith('V') or str(code).startswith('E'):
              return 'Other'
          c = float(code)
      except ValueError:
          return 'Other'
      if 390 <= c <= 459 or c == 785:
          return 'Circulatory'
      if 460 <= c <= 519 or c == 786:
          return 'Respiratory'
      if 520 <= c <= 579 or c == 787:
          return 'Digestive'
      if str(code).startswith('250'):
          return 'Diabetes'
      if 800 <= c <= 999:
          return 'Injury'
      if 710 <= c <= 739:
          return 'Musculoskeletal'
      if 580 <= c <= 629 or c == 788:
          return 'Genitourinary'
      if 140 <= c <= 239:
          return 'Neoplasms'
      return 'Other'

  df['diag_1_group'] = df['diag_1'].apply(icd9_group)
  df = df.drop(columns=['diag_1'])

# Recode ? to unknown for remaining categorical vars - ease one-hot
  for c in df.select_dtypes(include='object').columns:
      df[c] = df[c].replace('?', 'Unknown')

# id's as strings
  for c in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
      df[c] = df[c].astype(str)

# Collapse the 23 indiv drug columns into summary of Up/Down/Steady
  med_cols = ['metformin','repaglinide','nateglinide','glimepiride','glipizide',
              'glyburide','pioglitazone','rosiglitazone','insulin','glyburide-metformin']

  df['num_med_changes'] = (df[med_cols].isin(['Up', 'Down'])).sum(axis=1)
  df['num_meds_steady'] = (df[med_cols] == 'Steady').sum(axis=1)

# Make <30 the target - now binary classification
  df['target'] = (df['readmitted'] == '<30').astype(int)

# Finally,
  print("Final shape:", df.shape)

  return df

In [304]:
# Clean the patient-level data for the Random Forest
df = clean_diabetes_data(f)

# IDs for exactly the patients that remain after cleaning
patient_ids = raw_ehr.loc[
    df.index,
    ["patient_nbr", "encounter_id"]
].copy()

print("Cleaned model data:", df.shape)
print("Original EHR data:", raw_ehr.shape)

Final shape: (69973, 33)
Cleaned model data: (69973, 33)
Original EHR data: (101766, 50)


In [305]:
# 2. Train/Test split and preprocessing

# target named
X = df.drop(columns=['target', 'readmitted'])
y = df['target']

# Column standardization
numeric_feats = ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
                  'num_medications', 'number_outpatient', 'number_emergency',
                  'number_inpatient', 'number_diagnoses',
                  'num_med_changes', 'num_meds_steady']  # engineered counts, were previously
                  # missing from this list and got one-hot encoded as categories instead of
                  # standardized like the other count features
categorical_feats = [c for c in X.columns if c not in numeric_feats]

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, patient_ids, test_size=0.2, stratify=y, random_state=18
)

ct = ColumnTransformer([
    ("standardize", StandardScaler(), numeric_feats),
    ("dummify", OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first'), categorical_feats)
])

# Training splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=18)

In [306]:
# Class imbalance
# for the model, class weight is notched to balanced
# so that minority class is not ignored by the model
# ROC AUC as the accuracy score

In [307]:
# Logistic regression - tells interpretable coefficients
best_log = Pipeline([
    ("preprocessing", ct),
    ("log_regression", LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1))
])

best_log.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('standardize',
                                                  StandardScaler(),
                                                  ['age', 'time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'number_inpatient',
                                                   'number_diagnoses',
                                                   'num_med_changes',
                                                   'num_meds_steady']),
                                                 ('dummify',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ig...
                                                   'admission_type_id',
                                                   'discharge_disposition_id',
                                                   'admission_source_id',
                                                   'max_glu_serum', 'A1Cresult',
                                                   'metformin', 'repaglinide',
                                                   'nateglinide', 'glimepiride',
                                                   'glipizide', 'glyburide',
                                                   'pioglitazone',
                                                   'rosiglitazone', 'insulin',
                                                   'glyburide-metformin',
                                                   'change', 'diabetesMed',
                                                   'diag_1_group'])])),
                ('log_regression',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=1000))])

In [308]:
# Random Forest Classifier

rf_pipeline = Pipeline([
    ("preprocessing", ct),
    ("rf_classifier", RandomForestClassifier(
        n_estimators=300, class_weight='balanced_subsample', random_state=18, n_jobs=-1))
])

param_rf = {
    'rf_classifier__max_depth': [8, 12, None],
    'rf_classifier__min_samples_leaf': [5, 20]
}
gscv_rf = GridSearchCV(rf_pipeline, param_rf, cv=cv, scoring='roc_auc', n_jobs=-1)
gscv_rf.fit(X_train, y_train)

best_rf = gscv_rf.best_estimator_
print("Best params:", gscv_rf.best_params_)
print("CV ROC AUC:", gscv_rf.best_score_)

Best params: {'rf_classifier__max_depth': None, 'rf_classifier__min_samples_leaf': 20}
CV ROC AUC: 0.6489153756273437


In [309]:
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

print("Test ROC AUC:", roc_auc_score(y_test, y_proba_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))

Test ROC AUC: 0.6468201917604308
Confusion matrix:
 [[10591  2149]
 [  827   428]]
Recall: 0.34103585657370517
Precision: 0.1660845944897167


In [310]:
# feature names after one-hot encoding- needed by predict_readmission_risk
feat_names = best_log.named_steps['preprocessing'].get_feature_names_out()
print("Ready:", X.shape, len(feat_names), "features after encoding")

Ready: (69973, 31) 108 features after encoding


# Why a Model Alone Is Not Enough

A readmission-risk score by itself is not actionable for a clinician. A probability number does not explain why a patient is high-risk, what evidence in the chart supports that risk, what current diabetes-care guidance says about this specific patient, or whether the care that guidance calls for has already happened.

Instead of returning a single output, the system requires a few additional items.

1. It needs to explain which patient-specific factors are driving the risk prediction.

2. It needs to connect the risk prediction to current, specific ADA guidance rather than general clinical knowledge.

3. It needs to compare guideline recommendations against what is actually documented in the patient's EHR.

4. It needs to package all of the collected information from the prevouis agents into one auditable summary a clinician can review and act on quickly. An added, educated opinion for the clincian to save time between patients.

#Agentic Architecture: Clinician Assistant

Multi-Agent Flow (Start to Finish)

1. Risk Prediction (pulling Random Forest Predictions)

to

2. Patient Context

to

3. Diabetes Guideline RAG (pulling ADA Guidelines 2026)

to

4. Care Gap

to

5. Clinical Summary

In [318]:
#!pip install -q anthropic
import anthropic

from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=api_key)

Agent 1: Risk Prediction

In [319]:
# Tool creation (model wrapped to tool)

def predict_readmission_risk(patient_data):
    """Predict 30-day readmission risk for one patient."""

    if isinstance(patient_data, dict):

        # Start with an empty row having the same columns and dtypes as X_test
        patient_df = pd.DataFrame(columns=X_test.columns).astype(X_test.dtypes)

        # Add the patient's values
        patient_df.loc[0] = pd.Series(patient_data)

    elif isinstance(patient_data, pd.DataFrame):

        patient_df = patient_data.copy()

    else:
        raise TypeError("patient_data must be a dictionary or DataFrame")

    # Make sure columns are in the same order
    patient_df = patient_df[X_test.columns]

    probability = best_rf.predict_proba(patient_df)[0, 1]

    if probability >= 0.50:
        risk = "High"
    elif probability >= 0.25:
        risk = "Medium"
    else:
        risk = "Low"

    return {
        "readmission_probability": round(float(probability), 4),
        "risk": risk
    }

In [320]:
# Sanity check for model
patient = X_test.iloc[[50]]
risk_sanity_check = predict_readmission_risk(patient)
print(risk_sanity_check)

{'readmission_probability': 0.6164, 'risk': 'High'}


In [321]:
### System prompt

RISK_AGENT_SYSTEM = """
You are a hospital readmission risk prediction agent.

Your job is to assess a diabetic patient's risk of being
readmitted to the hospital within 30 days.

You have access to a machine-learning prediction tool.

When patient data is provided:
1. Use the prediction tool.
2. Report the predicted probability.
3. Report the risk level.
4. Give a brief explanation of the result.

Do not invent patient information.
Do not make a medical diagnosis.
The machine-learning model's prediction is the basis for the risk result.
"""

risk_tool = {
    "name": "predict_readmission_risk",
    "description": "Predict the probability that a diabetic patient will be readmitted within 30 days.",
    "input_schema": {
        "type": "object",
        "properties": {
            "patient_data": {
                "type": "object",
                "description": "The patient's model input features."
            }
        },
        "required": ["patient_data"]
    }
}

In [322]:
### Agent call
def run_risk_agent(patient_data):

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=RISK_AGENT_SYSTEM,
        tools=[risk_tool],
        messages=[
            {
                "role": "user",
                "content": f"""
                Predict the 30-day readmission risk for this patient:

                {json.dumps(patient_data, default=str)}
                """
            }
        ]
    )

    # Check that Claude will use the prediction tool
    for block in response.content:

        if block.type == "tool_use":

            tool_result = predict_readmission_risk(
                block.input["patient_data"]
            )

            # Send the tool result back to Claude
            final_response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=500,
                system=RISK_AGENT_SYSTEM,
                tools=[risk_tool],
                messages=[
                    {
                        "role": "user",
                        "content": f"""
                        Predict the 30-day readmission risk for this patient:

                        {json.dumps(patient_data, default=str)}
                        """
                    },
                    {
                        "role": "assistant",
                        "content": response.content
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(tool_result)
                            }
                        ]
                    }
                ]
            )

            return final_response.content[0].text

In [323]:
# Sanity check for Agent 1
patient = X_test.iloc[0].to_dict()

agent1_demo_result = run_risk_agent(patient)

print(agent1_demo_result)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros


## 30-Day Readmission Risk Assessment

**Predicted Probability:** 28.41%

**Risk Level:** Medium

### Explanation:

This patient has a **medium risk** of being readmitted within 30 days. The prediction model estimates approximately a 28% chance of readmission based on the provided clinical and demographic data.

**Key factors that may contribute to this risk level:**

- **Very short hospital stay** (1 day) - may indicate incomplete treatment or stabilization
- **No diabetes medications prescribed** despite being a diabetic patient (diabetesMed = "No")
- **No medication changes made** during hospitalization (num_med_changes = 0)
- **Primary diagnosis in the circulatory system**, which can be complex and require careful follow-up
- **No prior healthcare utilization** (0 outpatient, emergency, or inpatient visits), which may suggest limited engagement with healthcare or a new diagnosis

The combination of a brief hospitalization without diabetes medication management, along with a circula

Agent 2: Patient Context

In [324]:
### Tool creation
def get_patient_context(patient_data):
    """
    Prepare original EHR patient information for the Patient Context agent.
    """

    if isinstance(patient_data, pd.DataFrame):
        patient_data = patient_data.iloc[0].to_dict()

    return patient_data


### System prompt
PATIENT_CONTEXT_SYSTEM = """
You are a Patient Context agent for a diabetes care workflow.

Your job is to review the patient's original EHR data and create
a concise, clinically useful patient context for downstream agents.

The original EHR may contain more information than the features
used by the readmission prediction model. Use relevant information
from the full EHR record.

Identify and organize:

1. Basic patient characteristics
   - Age
   - Gender
   - Race

2. Hospital encounter information
   - Admission type
   - Admission source
   - Discharge disposition
   - Time in hospital

3. Hospital utilization
   - Inpatient visits
   - Emergency visits
   - Outpatient visits
   - Previous utilization patterns

4. Diabetes-related information
   - Diabetes medications
   - Insulin use
   - Medication changes
   - Diabetes medication status
   - A1C information
   - Glucose information

5. Clinical complexity
   - Number of diagnoses
   - Number of medications
   - Number of procedures
   - Number of laboratory procedures
   - Relevant diagnosis information

6. Other clinically relevant EHR information
   - Include additional fields from the original EHR when
     they provide useful context for diabetes care or
     hospital care.

7. Important missing information
   - Clearly identify information that is unavailable,
     represented as missing, or cannot be determined.

IMPORTANT:
- Use only information contained in the EHR.
- Do not invent information.
- Do not assume that missing documentation means a treatment
  was not provided.
- Do not diagnose the patient.
- Do not recommend treatment.
- Do not make guideline judgments. The ADA Guideline RAG
  agent will handle that separately.

Return a concise, structured patient context that can be passed
to the ADA Guideline RAG and Care Gap agents.
"""

### Agent
patient_context_agent = {
    "name": "Patient Context",
    "description": "Creates a concise clinical context from patient data.",
    "system_prompt": PATIENT_CONTEXT_SYSTEM
}


### Agent call
def run_patient_context_agent(patient_data):

    context = get_patient_context(patient_data)

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=900,
        system=PATIENT_CONTEXT_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": f"""
                Create the patient context for this patient:

                {json.dumps(context, default=str)}
                """
            }
        ]
    )

    return response.content[0].text

In [325]:
# Sanity check
# Using the patient ID associated with X_test row 50
patient_nbr = ids_test.iloc[50]["patient_nbr"]

# Retrieve the complete original EHR record
patient_raw = raw_ehr[
    raw_ehr["patient_nbr"] == patient_nbr
].copy()

# Run Patient Context using the original EHR
context = run_patient_context_agent(patient_raw)

print(context)

# PATIENT CONTEXT

## 1. Basic Patient Characteristics
- **Age**: 90-100 years
- **Gender**: Female
- **Race**: Caucasian
- **Weight**: Not documented (?)

## 2. Hospital Encounter Information
- **Admission Type**: Emergency (ID: 1)
- **Admission Source**: Emergency Room (ID: 7)
- **Discharge Disposition**: Discharged to home (ID: 3)
- **Time in Hospital**: 2 days
- **Payer**: Medicare (MC)
- **Medical Specialty**: Internal Medicine

## 3. Hospital Utilization
- **Current Encounter**: Emergency admission via ER
- **Number of Inpatient Visits**: 3 (including current)
- **Number of Emergency Visits**: 0 (in past year, excluding current)
- **Number of Outpatient Visits**: 0 (in past year)
- **Utilization Pattern**: History of inpatient hospitalizations; no recent emergency or outpatient visits documented

## 4. Diabetes-Related Information

### Medications:
- **Active Diabetes Medication**: Glyburide (status: Steady)
- **Insulin Use**: No
- **Other Diabetes Medications**: None documented


Agent 3: ADA Diabetic Care Guideline Agent

In [326]:
# First, RAG Model for Retrieving ADA Guidelines

#!pip install -q pypdf scikit-learn
import re
import json
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PDF_PATH = "/content/diabetes_careinfo_2026.pdf"

reader = PdfReader(PDF_PATH)

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    pages.append({
        "page": page_number,
        "text": text
    })

print("Pages loaded:", len(pages))

# Combine the PDF into chunks (~1 page/chunk)
chunks = []

for page in pages:
    text = page["text"]

    # Split long pages into smaller pieces
    pieces = re.split(r"\n\s*\n", text)

    for piece in pieces:
        piece = piece.strip()

        if len(piece) > 100:
            chunks.append({
                "page": page["page"],
                "text": piece
            })

print("Chunks created:", len(chunks))

vectorizer = TfidfVectorizer(
    stop_words="english"
)

chunk_matrix = vectorizer.fit_transform(
    [chunk["text"] for chunk in chunks]
)

def retrieve_ada_guidelines(query, top_k=5):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        chunk_matrix
    )[0]

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:
        results.append({
            "page": chunks[index]["page"],
            "text": chunks[index]["text"],
            "similarity": round(float(scores[index]), 4)
        })

    return results

Pages loaded: 17
Chunks created: 18


In [327]:
# Sanity check
retrieved_chunks = retrieve_ada_guidelines(
    "hospital discharge planning and preventing readmission in patients with diabetes"
)

for chunk in retrieved_chunks:
    print("\nPAGE:", chunk["page"])
    print(chunk["text"][:500])


PAGE: 12
illness on blood glucose levels, and the 
individual’s circumstances, capabilities, 
and preferences as well as the facility- 
related capabilities to manage diabetes 
(21,186–188). See section 13, “Older 
Adults,” for more information.
An outpatient follow-up visit with primary 
care, endocrinology, or a diabetes care 
and education specialist within 1 month 
of discharge is advised for all individuals 
experiencing hyperglycemia and/or hypo-
glycemia in the hospital. If glycemic man-
agement m

PAGE: 17
in type 1 ver sus type 2 diabetes with diabetic 
k etoacidosis: a rev iew and a propensity -mat ched 
nationwide analysis. J In vestig Med 2021;69: 
1196 – 1200
176. Shaka H, El-Amir Z, Wani F, et al. 
Hospitali zations and inpatient mortal ity f or 
h yperosmolar hyper glycemic st ate over a decade. 
Diabete s Res Clin Pract 2022;185:109230
177. Shand JAD, Morrow P , Braatvedt G. 
Mort ality after dischar ge from hospital f ollowing 
an episode of diabetic k etoacidosis. Ac

In [328]:
### System prompt
GUIDELINE_RAG_SYSTEM = """
You are the ADA 2026 Diabetes Guideline RAG agent.

Your job is to identify ADA 2026 recommendations that are
relevant to a specific hospitalized diabetes patient.

You have access to a retrieval tool containing the attached
ADA 2026 hospital diabetes guideline.

Always retrieve relevant guideline passages before answering.

For each relevant recommendation:

1. State the recommendation clearly.
2. Explain why it may apply to this patient.
3. Identify the ADA recommendation number when available.
4. Identify the source page when available.
5. Distinguish clearly between:
   - what the ADA recommends
   - what is known about the patient
   - what information is missing

Do not invent recommendations.
Do not claim that a patient received a treatment unless
that information is explicitly present in the patient data.

Your output will be passed to a Care Gap agent, so be
specific and structured.

Do not make a diagnosis or independently recommend treatment.
Use the ADA 2026 document as the authoritative source.
"""

### Tool definition
guideline_tool = {
    "name": "retrieve_ada_guidelines",
    "description": (
        "Searches the attached ADA 2026 Diabetes Care in the "
        "Hospital guideline and returns the most relevant passages."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Clinical question or guideline topic to search for."
            },
            "top_k": {
                "type": "integer",
                "description": "Number of guideline passages to retrieve."
            }
        },
        "required": ["query"]
    }
}

### Agent call
def run_guideline_rag_agent(patient_context):

    user_message = f"""
Review the following patient context and identify the most
relevant ADA 2026 hospital diabetes guidelines.

PATIENT CONTEXT:
{patient_context}

Use the guideline retrieval tool before answering.

Focus on recommendations that can potentially be compared
with the patient's actual EHR information.
"""

    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        system=GUIDELINE_RAG_SYSTEM,
        tools=[guideline_tool],
        messages=messages
    )

    while response.stop_reason == "tool_use":

        # Claude's tool requests
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_results = []

        # Handle every tool request
        for block in response.content:

            if block.type == "tool_use":

                retrieved = retrieve_ada_guidelines(
                    query=block.input["query"],
                    top_k=block.input.get("top_k", 5)
                )

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(
                        retrieved,
                        default=str
                    )
                })

        # Tool results must immediately follow tool_use
        messages.append({
            "role": "user",
            "content": tool_results
        })

        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1500,
            system=GUIDELINE_RAG_SYSTEM,
            tools=[guideline_tool],
            messages=messages
        )

    # Extract final text
    text_blocks = [
        block.text
        for block in response.content
        if block.type == "text"
    ]

    return "\n".join(text_blocks)

In [329]:
# Sanity check
patient = X_test.iloc[50].to_dict()

patient_context = run_patient_context_agent(patient)

guideline_result = run_guideline_rag_agent(
    patient_context
)

print(guideline_result)

Now I have comprehensive guideline information. Let me analyze the most relevant ADA 2026 recommendations for this patient.

---

# ADA 2026 HOSPITAL DIABETES GUIDELINE REVIEW

## PATIENT SUMMARY
95-year-old female with diabetes, 2-day hospitalization, discharged to SNF, taking only glyburide (sulfonylurea), no A1C or glucose monitoring documented, history of recurrent admissions, no outpatient follow-up documented.

---

## RELEVANT ADA 2026 RECOMMENDATIONS

### 1. **A1C TESTING AT ADMISSION**

**ADA Recommendation 16.1:**
"Perform an A1C test on all people with diabetes or hyperglycemia (random blood glucose >140 mg/dL [>7.8 mmol/L]) at the time of admission to the hospital if no A1C test result is available from the prior 3 months." (Grade B)

- **Source:** Page 1
- **Why this applies:** Patient has documented diabetes and was hospitalized
- **What is known:** A1C was not documented during this hospitalization
- **Gap identified:** No evidence that A1C testing was performed despite 

Agent 4: Care Gap Agent

In [330]:
### System prompt
CARE_GAP_SYSTEM = """
You are a Care Gap agent in a diabetes hospital-care workflow.

Your job is to compare the patient's ORIGINAL EHR information
with relevant ADA 2026 guideline recommendations.

For each relevant recommendation, classify the patient's care
using exactly one of these statuses:

MET
The EHR contains evidence that the recommended care was
provided or documented.

NOT MET
The EHR contains evidence that the recommended care was
not provided, or contains explicit evidence inconsistent
with the recommendation.

UNKNOWN
The available EHR information does not contain enough
evidence to determine whether the recommendation was met.

CRITICAL RULE:

Missing information is NOT the same as evidence that care
was not provided.

For example:

If the EHR contains no documentation of diabetes education,
the status should generally be UNKNOWN, not NOT MET.

Only use NOT MET when the EHR provides actual evidence that
the recommended care was not provided or was inconsistent
with the recommendation.

For every relevant recommendation, provide:

1. ADA recommendation
2. Status: MET / NOT MET / UNKNOWN
3. EHR evidence
4. Explanation
5. Missing information, if applicable

Only identify care gaps that are relevant to this patient.

Use the ADA guideline findings provided by Agent 3 as the
source of guideline recommendations.

Do not invent EHR information.
Do not invent guideline recommendations.
Do not diagnose the patient.
Do not independently recommend treatment.

If the EHR does not contain enough information to evaluate
a recommendation, explicitly say so.
"""

### Agent call
def run_care_gap_agent(patient_context, guideline_result):

    prompt = f"""
Compare this patient's ORIGINAL EHR information with the
relevant ADA 2026 guideline findings.

PATIENT EHR CONTEXT:
{patient_context}

ADA 2026 GUIDELINE FINDINGS:
{guideline_result}

Determine whether each relevant recommendation is:

MET
NOT MET
or
UNKNOWN

Remember:

MET = EHR evidence shows the care occurred.

NOT MET = EHR evidence shows the care did not occur
or was inconsistent with the recommendation.

UNKNOWN = the EHR does not contain enough information
to determine whether the care occurred.

Do not treat missing documentation as proof that care
was not provided.

Return a structured care-gap assessment.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        system=CARE_GAP_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text

In [331]:
patient_nbr = ids_test.iloc[50]["patient_nbr"]

# Full raw EHR record for this patient — the Patient Context agent is
# instructed to use the full record (not just the model's input features),
# so it needs the same raw lookup used in the Cell 33 sanity check.
patient_raw = raw_ehr[raw_ehr["patient_nbr"] == patient_nbr].copy()

# Model-ready feature row, used only for the ML risk model
patient_features = X_test.iloc[50].to_dict()

# Agent 1: Risk Prediction
risk_result = run_risk_agent(patient_features)

# Agent 2: Patient Context (uses the full raw EHR record)
patient_context = run_patient_context_agent(patient_raw)

# Agent 3
guideline_result = run_guideline_rag_agent(
    patient_context
)

# Agent 4
care_gap_result = run_care_gap_agent(
    patient_context,
    guideline_result
)

print("Agent 1: Risk Prediction")
print(risk_result)
print()
print("Agent 4: Care Gap Assessment")
print(care_gap_result)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros


Agent 1: Risk Prediction
## 30-Day Readmission Risk Assessment

**Predicted Probability:** 62.34%

**Risk Level:** **High**

### Explanation

This 95-year-old female patient has a **high risk** of being readmitted within 30 days. The model predicts approximately a 62% probability of readmission.

**Key factors contributing to this elevated risk include:**

- **Advanced age (95 years)** - Elderly patients typically face higher readmission rates
- **Recent hospitalization history** - 3 prior inpatient visits indicate complex medical needs
- **Short hospital stay (2 days)** - Brief hospitalizations may indicate incomplete treatment or stabilization
- **High complexity case** - 65 lab procedures, 20 medications, and 9 diagnoses suggest significant medical complexity
- **No medication changes** - Despite hospitalization, no adjustments were made to the diabetes treatment regimen
- **Multiple comorbidities** - 9 diagnoses indicate complex health conditions beyond diabetes

**Clinical Recomme

Agent 5: Clinical Summary

In [332]:
### System prompt
CLINICAL_SUMMARY_SYSTEM = """
You are the Clinical Summary agent in a diabetes hospital-care
workflow.

Your job is to synthesize the outputs of the previous agents into
one concise, structured clinical summary.

You will receive:

1. Risk Prediction
2. Patient Context
3. ADA 2026 Guideline Findings
4. Care Gap Assessment

Your summary should include:

1. Patient overview
   - Important demographic and clinical characteristics
   - Relevant hospitalization information

2. Readmission risk
   - Report the risk prediction provided by Agent 1
   - Do not recalculate or change the prediction

3. Clinical context
   - Summarize the most relevant diabetes and hospitalization
     information from Agent 2

4. Guideline considerations
   - Summarize the most relevant ADA 2026 recommendations
   - Do not introduce recommendations that were not identified
     by the Guideline RAG agent

5. Care gaps
   - Highlight the most important MET, NOT MET, and UNKNOWN
     findings from Agent 4
   - Give particular attention to documented NOT MET findings
   - Clearly distinguish UNKNOWN from NOT MET

6. Overall summary
   - Provide a concise synthesis of the patient's situation
   - Identify the most important documented care gaps or
     information gaps

IMPORTANT:
- Use only information provided by the previous agents.
- Do not invent clinical information.
- Do not diagnose the patient.
- Do not recalculate risk.
- Do not create new treatment recommendations.
- Do not turn UNKNOWN findings into NOT MET findings.
- Do not claim that care occurred unless supported by the
  provided EHR information.

The final output should be concise, structured, and suitable
for a clinician reviewing the patient.
"""

### Agent call
def run_clinical_summary_agent(
    risk_result,
    patient_context,
    guideline_result,
    care_gap_result
):

    prompt = f"""
Create the final clinical summary using the outputs from
the previous agents.

RISK PREDICTION
{risk_result}

PATIENT CONTEXT
{patient_context}

ADA 2026 GUIDELINES
{guideline_result}

CARE GAP ASSESSMENT
{care_gap_result}

Synthesize these findings into one concise clinical summary.

Do not add information that is not present in these outputs.
Do not make new clinical recommendations.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        system=CLINICAL_SUMMARY_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text

In [333]:
# Test
clinical_summary = run_clinical_summary_agent(
    risk_result,
    patient_context,
    guideline_result,
    care_gap_result
)

print(clinical_summary)

# CLINICAL SUMMARY

## Patient Overview
- **Demographics:** Female, age 90-100 years, Caucasian
- **Encounter:** Emergency admission via Emergency Room for iron deficiency anemia (primary diagnosis), with secondary diagnoses including urinary tract disorder and disseminated candidiasis
- **Hospital Stay:** 2 days, Internal Medicine service
- **Discharge:** To Skilled Nursing Facility
- **Clinical Complexity:** 9 diagnoses, 20 medications, 65 laboratory procedures performed
- **Hospital Utilization:** Third inpatient visit; no documented emergency or outpatient visits in past year

---

## 30-Day Readmission Risk

**Predicted Probability: 62.34% (HIGH RISK)**

Key contributing factors to elevated risk:
- Advanced age (90-100 years)
- Recent hospitalization history (3 prior inpatient visits)
- Short hospital stay (2 days)
- High medical complexity (65 lab procedures, 20 medications, 9 diagnoses)
- No medication changes despite hospitalization
- Multiple comorbidities
- Discharge to SNF r

#Evaluation of the Clincian Assistant System

Prediction Model Performance

Observed: Test ROC AUC 0.647, Recall 0.341, Precision 0.166. At the 0.5 threshold: 428 true positives, 2,149 false positives, 827 false negatives. Roughly 5 of 6 patients flagged high-risk are false positives; two-thirds of actual 30-day readmissions are missed.

Not observed: No signal from clinical notes, vitals trends, social determinants, or post-discharge circumstances — none are present in the source dataset, and none are recoverable through better modeling of the fields that are present. Strack et al. (2014), the paper introducing this dataset, reports comparable discrimination on the same task, which rules out an implementation defect as the explanation for the ceiling.

Implication: the model is not fit to act as a decision gate. It functions correctly as a directional input to Agent 5, provided nothing downstream treats its output as ground truth. Any future use of this score to trigger an automated action (discharge hold, alert, etc.) would need materially better precision than this to justify the false-positive rate.set does not include.

Agent System Performance

Observed: the Care Gap agent held its MET/NOT MET/UNKNOWN distinction across test runs — missing documentation was consistently classified UNKNOWN, not NOT MET, matching the constraint in its system prompt. The Guideline RAG agent grounded every recommendation in a page-cited passage from the source PDF; no guideline content was invented.

Not observed: none of the agents were given a lookup table for the dataset's coded categorical fields (admission_type_id, discharge_disposition_id, admission_source_id). Patient 50's discharge_disposition_id is 3 — "Discharged/transferred to SNF" per the dataset's codebook. One test run of the Patient Context agent reported this patient "discharged to home"; a separate run on the same code reported "skilled nursing facility." The agent is inferring meaning from a bare integer rather than reading a decode table, and produces different answers on different runs.

Implication: the claim "no agent fabricated patient data" does not hold for coded fields. It holds for free-text and directly-labeled fields only. This is a scoped hallucination surface, not a general reliability failure — but it's a live one, and it reached the final Clinical Summary output undetected. Fix before this is treated as validated: supply a decode table for every coded field in the prompt context, or pre-decode these fields in clean_diabetes_data() before they ever reach an agent.

#Production Considerations

Security and Compliance

Observed: the current build uses de-identified public research data and a direct API call from a notebook. There is no BAA sign-off, no compliant logging layer, no secrets manager.

Not observed: This is expected at prototype stage and is not a defect in this build; it's a defined gap between prototype and pilot.

Implication: none of this code is deployable against real PHI as-is. A production version requires: a HIPAA-eligible execution environment, a signed BAA covering the specific LLM API use, prompt/response logging either disabled or routed through a compliant layer, and an audit trail (prompt, model, response, timestamp, reviewer disposition) for every agent output tied to a real patient.

Reliability and Safety

Observed: each agent's system prompt encodes its own grounding constraints (don't invent, don't diagnose, don't recommend treatment) and those constraints held on non-coded fields across test runs.

Not observed: no independent verification step confirms that a downstream agent's claims actually trace to an upstream agent's output. No retry/fallback path exists for tool-use failures or malformed responses; the pipeline assumes every API call succeeds. No confidence or coverage signal exists on the RAG layer, so a weak-match retrieval (low cosine similarity, no relevant chunk found) is indistinguishable downstream from a strong one.

Implication: the system's failure mode is not obvious. It does not have a state that signals "I don't have enough information to be sure of this" beyond what the model's prose voluntarily includes. A wrong output looks identical in format to a correct one at every stage. This is a large gap between this build and something a clinician could rely on.

Scalability

Observed: RAG retrieval runs on TF-IDF over 18 chunks from one 17-page PDF, chunked on double-newline splits. Pipeline execution is sequential per patient — five agents, five round trips, no batching.

Not observed: performance or cost at volume. This has not been tested past single-patient runs.

Implication: the current retrieval approach is a valid prototype-scale choice and would not need to change for a single-guideline pilot. It does not extend to the full ADA Standards of Care or hospital-specific policy documents without a real vector store and a chunking strategy that isn't newline-dependent. The sequential per-patient execution would not survive hospital-scale volume without batching, async execution, or an event-driven trigger on the discharge workflow.

Readiness

This is a single-patient pilot demonstration, not a production candidate. No item in this section has been implemented: all are documented gaps to close before a broader pilot, not completed mitigations.

#Recommendation and Next Steps

System Recommendation

Claude Sonnet was used as the default model for every agent in this build. At time of build, it reported 77.2% on SWE-bench Verified and 61.4% on OSWorld, at roughly a fifth the price of Anthropic's prior top-tier model. For a pipeline making five sequential tool-augmented calls per patient, that cost-to-capability ratio was the deciding factor — strong instruction-following and tool-use reliability, which the strict MET/NOT MET/UNKNOWN constraints in this system depend on, without paying reasoning-model pricing on every call. This was not benchmarked against alternatives within this project; it's a reasoned default, not a validated one (see below).

Next Steps

1. Fix the coded-field hallucination gap. Supply a decode table for admission_type_id, discharge_disposition_id, and admission_source_id in agent context, or resolve these to labels in clean_diabetes_data() before any agent sees them. This is the highest-priority fix as it is the one confirmed correctness failure in the current outputs.

2. Add a grounding-verification step. An explicit check — agent or deterministic — that each claim in the Care Gap and Clinical Summary outputs traces to text actually present in the upstream outputs, rather than relying on prompt instruction alone.

3. Model routing. Not every agent needs the same capability tier. Patient Context and Care Gap are largely structured extraction/comparison and are candidates for a smaller, cheaper model; Guideline RAG reasoning and Clinical Summary synthesis are the higher-value targets for the stronger model.

4. Token cost reduction. The pipeline currently re-sends full patient context and guideline text on every downstream call. Cache retrieved guideline chunks instead of re-fetching, compress patient context to a structured schema before passing to Agents 3–5, and benchmark prompt-caching on the system prompts, which are static per patient and currently resent on every call.

5. Cross-model evaluation. Benchmark this pipeline against at least one other provider's model on identical patients, scored on grounding accuracy, guideline-citation correctness, and cost per patient, to validate model choice rather than assume it.

6. Clinician feedback loop. This case study had no clinician input at any stage. This is an acknowledged scope limitation, not an oversight. Direct feedback from practicing clinicians at a sample of the source hospitals would be the next step to move prompt design from internally-reasoned to externally-validated.

###Resources



Hospital EHR Data (UC Irvine): https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

Diabetic Care Guidelines (ADA): https://watermark02.silverchair.com/dc26s016.pdf?token=AQECAHi208BE49Ooan9kkhW_Ercy7Dm3ZL_9Cf3qfKAc485ysgAAA0kwggNFBgkqhkiG9w0BBwagggM2MIIDMgIBADCCAysGCSqGSIb3DQEHATAeBglghkgBZQMEAS4wEQQMe5g2rB1pCIHdDv6qAgEQgIIC_GHcB-WqwJlVQtGgCKLIOqTBEAO9v5kPDbIPJg-9h4V9f2sioFLAGmdx2Bx1RjQMneACn7y8Z6nsNcnlzmiDDFIHGfAq_5iDGWyP75b3CqPuMH_EP5fH1h7m8JzCrnM5js_bm98pqkEg7c3Tt4suRfWXw8hTxssmpQo7eAHRGWgHqmyOnQZw9JmL1CEgjHOQf2OPhnIdTYrr2nZutwxwZ9b1GCcTVAxcPjIQCDhRTGrLVCBgHDqJb_gfzYtKxQDkinTUKl2ABlO2-DruOHn7Ip8VlpVaVWmaUq4F_rw9LiEDJPFIIiDZIt2ePIoOlXtinZqBRHIMfwtFJIdkOrouyvYsrO2TgbU_X4ZtOr4XvuEVvk4dZAhaiS1kXAvlIu2UdAhGFEFAlm8lnKnGjvFV33ZGPyWjxys-zcZYlkjXCStY4G9tiGFDYn79xWHjSeRePjZJKPL19XIBNbEAHAkl7x3yzBa1dvCKgQITgugXg_6kJz8umWWDo0G0PucdSoug4FCagzVTfet10ksdDc0vVTl9YxftPp46mM3m-IXgYoaaErbvgxqKB2gKf564N6C6X_iwLzuv67Eq4C0GiiSk4EZtp934-pjZjESEb-ezkxD1CyoLqbLj3T9tiWODzCkmnVsMffZn2PePi7Khqm-WsziWeuqUpZxFWJGXiKOU-zSU6GrJ87Fn7Nfte46wo0Bfjf9PwXUd4DAxrKXXTkF3kUhNmm4BoAYvBLWUT18cLUJPlEpJM-LtyoFr25ASD9UAybMeSMlOuIe79EjHCbhWLe3vpDC9nqfypIHqqPAZhjO2s7pKRwMRaF6PQnnT0mSSDqgF_3JalJJL2f05Pm2YyjSum_As6Ca2zjZ5aKRXm4o-4lKokhbmCNlXacwm4wVRxvt1a57SQX-W1DIzGUyklne2sEPMX4Qb8n_VxTeSQT6yvAxJideWaFYgVR_7vtvpz1hiCPF14NyTzHphwahrL_HtbFXrL4kEu5tDnK7VtXTZwJc30xTopvzdlkTg

Anthropic Performance: https://www.anthropic.com/news/claude-sonnet-4-5

LLMs: Claude and ChatGPT